# World Bank Indicators API Example

> The following is an example of a [Jupyter notebook](https://jupyter.org) - a tutorial of how to retrieve data from the [World Bank Indicators API](https://datahelpdesk.worldbank.org/knowledgebase/articles/889392-about-the-indicators-api-documentation) - that illustrates how to use computational content with the [template](https://worldbank.github.io/template). 

print("hello")

In [2]:
import pandas as pd

# Load the Piece 1 output file
FILE = "open_router_data_classified.csv"   # change if needed
df = pd.read_csv(FILE)

# --------------------------------------------------
# 1. Overall classification counts
# --------------------------------------------------

# --------------------------------------------------
# 2. Flag counts
# --------------------------------------------------
flag_cols = [
    "is_open_source",
    "is_chinese_model",
    "is_unknown_country",
    "is_unclassified_model",
    "is_missing_model_slug",
]

# --------------------------------------------------
# 3. Rows and token share by classification status
# --------------------------------------------------
summary = pd.DataFrame({
    "rows": df["classification_source"].value_counts(dropna=False),
    "tokens": df.groupby("classification_source", dropna=False)["total_tokens"].sum()
}).fillna(0)

summary["row_pct"] = (summary["rows"] / len(df) * 100).round(2)
summary["token_pct"] = (summary["tokens"] / df["total_tokens"].sum() * 100).round(2)

# --------------------------------------------------
# 4. Unique model counts
# --------------------------------------------------
model_summary = (
    df.groupby("classification_source", dropna=False)["model_permaslug"]
      .nunique()
      .rename("unique_models")
      .reset_index()
      .sort_values("unique_models", ascending=False)
)


# --------------------------------------------------
# 5. Top unclassified models by rows and tokens
# --------------------------------------------------
unclassified = df[df["classification_source"] == "unclassified"].copy()

if len(unclassified) != 0:
    top_unclassified = (
        unclassified.groupby("model_permaslug", dropna=False)
        .agg(
            rows=("model_permaslug", "size"),
            total_tokens=("total_tokens", "sum"),
            countries=("country", "nunique"),
            months=("month", "nunique"),
        )
        .reset_index()
        .sort_values(["total_tokens", "rows"], ascending=[False, False])
    )


# --------------------------------------------------
# 6. Top missing / unknown data issues
# --------------------------------------------------
unknown_country = df[df["is_unknown_country"] == True]

missing_slug = df[df["is_missing_model_slug"] == True]

# --------------------------------------------------
# 7. Quick monthly view of classified vs unclassified
# --------------------------------------------------
monthly_status = (
    df.assign(
        status=df["classification_source"].where(
            df["classification_source"].isin(["unclassified", "missing_slug"]),
            "classified"
        )
    )
    .groupby(["month", "status"], dropna=False)
    .agg(
        rows=("status", "size"),
        total_tokens=("total_tokens", "sum")
    )
    .reset_index()
    .sort_values(["month", "status"])
)

# --------------------------------------------------
# 8. Optional: save review tables
# --------------------------------------------------
summary.to_csv("classification_summary.csv")
model_summary.to_csv("classification_unique_models.csv", index=False)
monthly_status.to_csv("classification_monthly_status.csv", index=False)

if len(unclassified) > 0:
    top_unclassified.to_csv("top_unclassified_models.csv", index=False)


In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.express as px
import pycountry

# ----------------------------
# Parameters
# ----------------------------
INPUT_FILE = "country_month_indices.csv"
OUTPUT_DIR = Path("piece4_outputs")
TOP_N_COUNTRIES_FOR_LINES = 15
EXCLUDE_UNKNOWN_COUNTRY = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------
# Load and validate
# ----------------------------
df = pd.read_csv(INPUT_FILE)

required_cols = {
    "country",
    "month",
    "total_tokens",
    "token_index",
    "open_source_index",
    "chinese_index",
}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

# ----------------------------
# Clean fields
# ----------------------------
df = df.copy()

df["country"] = (
    df["country"]
    .fillna("UNK")
    .astype(str)
    .str.strip()
    .str.upper()
    .replace("", "UNK")
)

df["month"] = df["month"].fillna("").astype(str).str.strip()

unique_months = pd.Series(df["month"].unique(), name="month")
month_lookup = pd.Series(
    pd.to_datetime(unique_months, errors="coerce").dt.to_period("M").values,
    index=unique_months
)
df["month_period"] = df["month"].map(month_lookup)

for col in ["total_tokens", "token_index", "open_source_index", "chinese_index"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["total_tokens"] = df["total_tokens"].fillna(0)
df.loc[df["total_tokens"] < 0, "total_tokens"] = 0

for col in ["token_index", "open_source_index", "chinese_index"]:
    df[col] = df[col].fillna(0).clip(lower=0, upper=100)

df = df.dropna(subset=["month_period"]).copy()

if EXCLUDE_UNKNOWN_COUNTRY:
    df = df.loc[df["country"] != "UNK"].copy()

if df.empty:
    raise ValueError("No rows available for visualisation after cleaning/filtering.")

df = df.sort_values(["month_period", "country"]).reset_index(drop=True)
df["month_label"] = df["month_period"].astype(str)

# ----------------------------
# ISO3 helper
# ----------------------------
def iso2_to_iso3(code: str):
    code = (code or "").strip().upper()

    manual = {
        "UK": "GBR",
        "EL": "GRC",
        "XK": "XKX",
    }
    if code in manual:
        return manual[code]

    if len(code) == 3 and code.isalpha():
        return code

    if len(code) != 2 or not code.isalpha():
        return None

    match = pycountry.countries.get(alpha_2=code)
    if match:
        return match.alpha_3

    fallback = {
        "US": "USA", "GB": "GBR", "DE": "DEU", "FR": "FRA", "IT": "ITA",
        "ES": "ESP", "PT": "PRT", "NL": "NLD", "BE": "BEL", "CH": "CHE",
        "AT": "AUT", "SE": "SWE", "NO": "NOR", "DK": "DNK", "FI": "FIN",
        "IE": "IRL", "PL": "POL", "CZ": "CZE", "HU": "HUN", "RO": "ROU",
        "BG": "BGR", "GR": "GRC", "TR": "TUR", "UA": "UKR", "RU": "RUS",
        "CA": "CAN", "MX": "MEX", "BR": "BRA", "AR": "ARG", "CL": "CHL",
        "CO": "COL", "PE": "PER", "AU": "AUS", "NZ": "NZL", "JP": "JPN",
        "CN": "CHN", "HK": "HKG", "TW": "TWN", "IN": "IND", "ID": "IDN",
        "MY": "MYS", "SG": "SGP", "TH": "THA", "VN": "VNM", "PH": "PHL",
        "KR": "KOR", "ZA": "ZAF", "NG": "NGA", "KE": "KEN", "GH": "GHA",
        "ET": "ETH", "TZ": "TZA", "UG": "UGA", "MW": "MWI", "ZM": "ZMB",
        "ZW": "ZWE", "MZ": "MOZ", "BW": "BWA", "NA": "NAM", "EG": "EGY",
        "MA": "MAR", "TN": "TUN", "DZ": "DZA", "AE": "ARE", "SA": "SAU",
        "IL": "ISR",
    }
    return fallback.get(code)

df["iso3"] = df["country"].map(iso2_to_iso3)

map_df = df.dropna(subset=["iso3"]).copy()
map_df = map_df.sort_values(["month_label", "country"]).reset_index(drop=True)

top_countries = (
    df.groupby("country", dropna=False)["total_tokens"]
    .sum()
    .sort_values(ascending=False)
    .head(TOP_N_COUNTRIES_FOR_LINES)
    .index.tolist()
)

line_df = df.loc[df["country"].isin(top_countries)].copy()
line_df = line_df.sort_values(["month_period", "country"]).reset_index(drop=True)

if line_df.empty:
    raise ValueError("No countries available for line charts after selecting top countries.")

country_iso_lookup = (
    df[["country", "iso3"]]
    .drop_duplicates()
    .sort_values(["country", "iso3"], na_position="last")
)

In [4]:
def make_line_chart(data: pd.DataFrame, y_col: str, title: str, y_axis_title: str):
    fig = px.line(
        data,
        x="month_label",
        y=y_col,
        color="country",
        markers=True,
        title=title,
        hover_data={
            "country": True,
            "month_label": True,
            "total_tokens": ":,.0f",
            "token_index": ":.2f",
            "open_source_index": ":.2f",
            "chinese_index": ":.2f",
        },
        category_orders={"month_label": sorted(data["month_label"].unique())},
    )

    fig.update_layout(
        xaxis_title="Month",
        yaxis_title=y_axis_title,
        legend_title_text="Country",
        hovermode="x unified",
    )
    return fig

fig_token = make_line_chart(
    line_df,
    y_col="token_index",
    title=f"Token volume index over time — top {TOP_N_COUNTRIES_FOR_LINES} countries by total tokens",
    y_axis_title="Token volume index (0–100)",
)

fig_open = make_line_chart(
    line_df,
    y_col="open_source_index",
    title=f"Open-source index over time — top {TOP_N_COUNTRIES_FOR_LINES} countries by total tokens",
    y_axis_title="Open-source index (0–100)",
)

fig_chinese = make_line_chart(
    line_df,
    y_col="chinese_index",
    title=f"Chinese model index over time — top {TOP_N_COUNTRIES_FOR_LINES} countries by total tokens",
    y_axis_title="Chinese index (0–100)",
)

fig_token.show()
fig_open.show()
fig_chinese.show()

fig_token.write_html(OUTPUT_DIR / "line_token_index.html", include_plotlyjs="cdn")
fig_open.write_html(OUTPUT_DIR / "line_open_source_index.html", include_plotlyjs="cdn")
fig_chinese.write_html(OUTPUT_DIR / "line_chinese_index.html", include_plotlyjs="cdn")

In [5]:
print("Hello")

Hello
